# 00 — Environment setup (EMR)

Create the Glue database and generate seed source files (products, customers, clickstream events). Port of `notebooks/00_environment_setup.py` — no `dbutils`, no wheel-install cell (the persistent EMR cluster's bootstrap action already installed `retail_lakehouse`), Glue database instead of Unity Catalog catalog/schema/volume.

Run the cell below first, before anything else -- it configures Delta Lake for this notebook's Spark session (EMR doesn't bundle Delta by default, unlike Databricks). `%%configure -f` must run before any other Spark code in this session.

In [ ]:
%%configure -f
{"conf": {"spark.jars.packages": "io.delta:delta-spark_2.12:3.1.0", "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension", "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"}}

In [ ]:
schema = "retail_lakehouse"
base_path = "s3://<your-lakehouse-bucket>/data"  # from `terraform output lakehouse_bucket_name`

from retail_lakehouse.config import PipelineConfig
cfg = PipelineConfig(schema=schema, base_path=base_path)

spark.sql(f"CREATE DATABASE IF NOT EXISTS `{cfg.schema}` LOCATION '{cfg.path('tables')}'")
spark.sql(f"USE `{cfg.schema}`")
spark.conf.set("spark.sql.shuffle.partitions", "8")
print("Spark version:", spark.version)

## Generate seed data

Products and customers are batch dimension data (as if loaded from an OLTP export). Events simulate what would otherwise arrive continuously from MSK in `02_kafka_msk_streaming_ingest.ipynb` — having a batch copy also lets `01` demonstrate the pure-batch path for comparison.

In [ ]:
from retail_lakehouse.generate import synthetic_products, synthetic_customers, synthetic_events
import random
random.seed(42)

product_rows = synthetic_products(50)
customer_rows = synthetic_customers(200)
event_rows = list(synthetic_events(2000))

product_df = spark.createDataFrame(product_rows)
customer_df = spark.createDataFrame(customer_rows)
event_df = spark.createDataFrame(event_rows)

product_df.write.mode("overwrite").format("delta").saveAsTable(cfg.table("dim_product_seed"))
customer_df.write.mode("overwrite").format("delta").saveAsTable(cfg.table("dim_customer_seed"))
event_df.write.mode("overwrite").json(cfg.path("source", "events_json"))
product_df.write.mode("overwrite").option("header", True).csv(cfg.path("source", "products_csv"))
customer_df.write.mode("overwrite").option("header", True).csv(cfg.path("source", "customers_csv"))

print("Created seed tables and files")
event_df.limit(10).show(truncate=False)

## Tables created so far

In [ ]:
spark.sql(f"SHOW TABLES IN `{cfg.schema}`").show()

## Next steps

- `01_batch_lakehouse_bronze_silver_gold.ipynb` — batch path from the seed files.
- `02_kafka_msk_streaming_ingest.ipynb` — real MSK ingestion (requires `infra/terraform` provisioned; see `RUNBOOK.md`).